# Ch03 CoT Reasoning — GPU Supplement

This notebook requires a CUDA GPU with ≥12 GB VRAM for larger reasoning models.

**Experiments**:
1. Reasoning depth comparison: deepseek-r1:7b vs deepseek-r1:14b
2. Overthinking demo: reasoning token waste on trivial problems

**Requirements**: Ollama + GPU, models: `ollama pull deepseek-r1:7b deepseek-r1:14b`

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected — run the CPU notebook instead: notebook-exercise.ipynb\n"
        "To provision a GPU machine: see notes/06-ai-infrastructure/ch01-gpu-architecture/"
    )

device = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {device}  |  VRAM: {vram:.1f} GB")

assert vram >= 12, f"Need ≥12 GB VRAM for reasoning models, got {vram:.1f} GB"
print("✓ GPU check passed")

## Experiment 1 — Reasoning Depth vs Model Size

Compare deepseek-r1:7b vs deepseek-r1:14b on an AIME-style multi-step problem.

**Hypothesis**: Larger reasoning model should generate longer, more detailed `<think>` traces and higher accuracy.

In [ ]:
import ollama
import re

problem = """Alice has 3 apples. She gives Bob half of them (rounded down), then buys 5 more. Bob gives her back 1. How many does Alice have?"""

models = ['deepseek-r1:7b', 'deepseek-r1:14b']
correct_answer = 8  # 3 - 1 + 5 + 1 = 8

for model in models:
    print(f"\n{'='*60}")
    print(f"Model: {model}")
    print('='*60)

    response = ollama.generate(
        model=model,
        prompt=problem,
        options={'temperature': 0.0}
    )

    full_text = response['response']

    # Extract <think> block if present
    think_match = re.search(r'<think>(.*?)</think>', full_text, re.DOTALL)
    if think_match:
        think_content = think_match.group(1)
        think_tokens = len(think_content.split())
        print(f"\n<think> block: {think_tokens} tokens")
        print(f"Sample: {think_content[:200]}...")
    else:
        think_tokens = 0
        print("\nNo <think> block found")

    # Extract final answer
    answer_match = re.search(r'(\d+)\s*apples?', full_text, re.IGNORECASE)
    if answer_match:
        answer = int(answer_match.group(1))
        correct = "✓" if answer == correct_answer else "✗"
        print(f"\nFinal answer: {answer} apples {correct}")
    else:
        print("\nCouldn't extract numerical answer")

## Experiment 2 — Overthinking on Trivial Problems

Test whether reasoning models waste tokens on problems that don't need reasoning.

**Hypothesis**: Reasoning models should use few/zero reasoning tokens on trivial problems like "What is 2+2?"

In [ ]:
import ollama
import re

problems = {
    'Trivial': "What is 2 + 2?",
    'Simple': "What is 12 × 15?",
    'Complex': "If 5 workers can build 5 widgets in 5 days, how many days does it take 10 workers to build 10 widgets?"
}

print("Testing deepseek-r1:14b reasoning token allocation:\n")

for complexity, problem in problems.items():
    response = ollama.generate(
        model='deepseek-r1:14b',
        prompt=problem,
        options={'temperature': 0.0}
    )

    full_text = response['response']

    # Extract <think> block
    think_match = re.search(r'<think>(.*?)</think>', full_text, re.DOTALL)
    if think_match:
        think_tokens = len(think_match.group(1).split())
    else:
        think_tokens = 0

    print(f"{complexity:>8} problem: {think_tokens:>4} reasoning tokens")
    print(f"  Q: {problem}")
    print(f"  A: {full_text.split('</think>')[-1].strip()[:100] if '</think>' in full_text else full_text[:100]}...")
    print()